# LargeGraphs direct layout demos

This notebook focuses on direct layout functions that return positioned nodes.
To render one of these layouts as an interactive graph, pass the returned nodes to `render(...)`.


In [ ]:
using Pkg
Pkg.activate(isdir(joinpath(pwd(), "src")) ? pwd() : joinpath(pwd(), ".."))
Pkg.instantiate()

using LargeGraphs
using Random

function medium_graph(node_count::Integer=100; seed::Integer=11)
    rng = MersenneTwister(seed)
    nodes = Vector{NamedTuple}(undef, node_count)
    for i in 1:node_count
        nodes[i] = (
            id=string(i),
            label=i <= 20 ? "node-$i" : nothing,
            size=1.2 + 1.6 * rand(rng),
            color=rand(rng, ("#2563eb", "#059669", "#d97706", "#7c3aed", "#db2777")),
        )
    end

    edges = NamedTuple[]
    seen = Set{Tuple{Int, Int}}()
    target_edges = 220
    while length(edges) < target_edges
        source = rand(rng, 1:node_count)
        target = rand(rng, 1:node_count)
        source == target && continue
        edge = source < target ? (source, target) : (target, source)
        edge in seen && continue
        push!(seen, edge)
        push!(edges, (
            source=string(source),
            target=string(target),
            size=0.6,
            color="#94a3b8",
        ))
    end

    nodes, edges
end

function large_graph(node_count::Integer=10_000, edge_count::Integer=30_000; seed::Integer=7)
    rng = MersenneTwister(seed)
    nodes = Vector{NamedTuple}(undef, node_count)
    for i in 1:node_count
        nodes[i] = (
            id=string(i),
            size=1.0 + 2.0 * rand(rng),
            color=rand(rng, ("#2563eb", "#0891b2", "#059669", "#d97706")),
            label=i <= 120 ? "node-$i" : nothing,
        )
    end

    edges = NamedTuple[]
    seen = Set{Tuple{Int, Int}}()
    while length(edges) < edge_count
        source = rand(rng, 1:node_count)
        target = rand(rng, 1:node_count)
        source == target && continue
        edge = source < target ? (source, target) : (target, source)
        edge in seen && continue
        push!(seen, edge)
        push!(edges, (
            source=string(source),
            target=string(target),
            size=0.5,
            color="#cbd5e1",
        ))
    end

    nodes, edges
end

medium_nodes, medium_edges = medium_graph()


## Direct layout functions on a medium graph (100 nodes)


### `random_layout`


In [ ]:
random_nodes = random_layout(medium_nodes; seed=11, extent=2.0)
random_nodes[1:8]

### `circular_layout`


In [ ]:
circular_nodes = circular_layout(medium_nodes; radius=2.0)
circular_nodes[1:8]

### `grid_layout`


In [ ]:
grid_nodes = grid_layout(medium_nodes; columns=10, spacing=0.5)
grid_nodes[1:8]

### `spectral_layout`


In [ ]:
spectral_nodes = spectral_layout(medium_nodes, medium_edges; extent=2.0, seed=5)
spectral_nodes[1:8]

### `force_directed_layout` with `algorithm=:fruchterman_reingold`


In [ ]:
fr_nodes = force_directed_layout(medium_nodes, medium_edges; algorithm=:fruchterman_reingold, iterations=120, seed=5, extent=2.0)
fr_nodes[1:8]

### `force_directed_layout` with `algorithm=:kamada_kawai`


In [ ]:
kk_nodes = force_directed_layout(medium_nodes, medium_edges; algorithm=:kamada_kawai, iterations=140, seed=5, extent=2.0)
kk_nodes[1:8]

### `force_directed_layout` with `algorithm=:forceatlas2`


In [ ]:
fa2_nodes = force_directed_layout(medium_nodes, medium_edges; algorithm=:forceatlas2, iterations=180, seed=5, extent=2.0)
fa2_nodes[1:8]

## Render one positioned layout

Direct layout functions only compute coordinates. This cell shows how to turn one of those coordinate sets into a rendered graph.


In [ ]:
render(fr_nodes, medium_edges; height="540px", hide_edges_on_move=true, label_density=0.7)
